# Join CL Federal Appellate Data to SCOTUS Scraped data

The SCOTUS Scraped data is used as a bridge to link the CL SCOTUS opinions to lower court opinions that appealed to SCOTUS. In order to establish the appellate linkage between SCOTUS and lower court opinions, we will leverage the docket number and the date to
1. Link the CL SCOTUS opinions to SCOTUS scraped data
2. Link the CL Lower Court opinions to SCOTUS scraped data
   - Link from Federal Appellate to SCOTUS
   - Link from other lower courts to SCOTUS

This notebook outlines the first bullet point of the second of the two steps above.

# Import Libaries

In [1]:
import os
import json

import numpy as np
import pandas as pd

from cl_utils import *

In [2]:
scotus = pd.read_json("data/scotus_scraped_data.json")
scotus.head()

,docket_number,filename,docket_date,case_title,lower_court,lower_court_case_numbers_raw,lower_court_case_numbers,lower_court_decision_date,lower_court_rehearing_denied_date,year,case_num
0,03-6084,03-6084.htm,2003-08-27,"Ronald Lee Smith, Petitioner v. Larry Reid, Wa...",United States Court of Appeals for the Tenth C...,(03-1016),03-1016,2003-05-16,2003-07-09,3,6084
1,03-10616,03-10616.htm,2004-05-28,"Jimmy Walker, Petitioner v. Florida","District Court of Appeal of Florida, Fourth Di...",(4D02-4272),4D02-4272,2004-04-21,None,3,10616
2,03-777,03-777.htm,2003-11-28,"Willie R. Flint, Petitioner v. ABB Inc., fka A...",United States Court of Appeals for the Elevent...,(02-15029),02-15029,2003-07-21,2003-08-27,3,777
3,03-10170,03-10170.htm,2004-05-05,"Loyda Lugones, Petitioner v. United States",United States Court of Appeals for the Elevent...,(02-12984),02-12984,2004-01-26,None,3,10170
4,03-8917,03-8917.htm,2004-02-19,"Ray Charles Smith, Petitioner v. United States",United States Court of Appeals for the Ninth C...,(03-10003),03-10003,2003-11-13,None,3,8917


## Remove the scotus records that do not have valid lower court info for appellate chain linking

In [3]:
scotus = scotus[(~scotus["lower_court"].isnull()) & (~scotus["lower_court_case_numbers"].isnull())]
len(scotus)

185843

# Load the courts data

In [4]:
courts_summary = pd.read_csv("data/scotus_lower_courts_mapped.csv")
len(courts_summary)

677

In [5]:
courts_summary.columns

Index(['lower_court', 'count', 'first_case_num', 'cl_court_id',
       'cl_court_name', 'manual', 'manual_type', 'auto_type', 'type'],
      dtype='object')

In [6]:
# Inspect records where there are more than one scotus lower court name to one cl_court_id
mapping_counts = courts_summary.groupby("cl_court_id")["lower_court"].nunique()
one_to_many_ids = mapping_counts[mapping_counts > 1].index
courts_summary[courts_summary["cl_court_id"].isin(one_to_many_ids)].groupby("cl_court_id").value_counts(["lower_court"])

cl_court_id  lower_court                                            
arizctapp    Arizona Court of Appeals                                   1
             Court of Appeals of Arizona, Division One                  1
             Court of Appeals of Arizona, Division Two                  1
calctapp     Court of Appeal of California, Fifth Appellate District    1
             Court of Appeal of California, First Appellate District    1
                                                                       ..
washctapp    Court of Appeals of Washington, Division 3                 1
wisctapp     Court of Appeals of Wisconsin, District I                  1
             Court of Appeals of Wisconsin, District II                 1
             Court of Appeals of Wisconsin, District III                1
             Court of Appeals of Wisconsin, District IV                 1
Name: count, Length: 84, dtype: int64

In [7]:
courts_summary = courts_summary[['lower_court', 'cl_court_id', 'cl_court_name', 'type']]
courts_summary.rename(columns={'lower_court': 'scotus_lower_court',
                               'cl_court_id': 'cl_lower_court_id',
                               'cl_court_name': 'cl_lower_court_name',
                               'type': 'cl_lower_court_type'}, inplace=True)
courts_summary.head()

,scotus_lower_court,cl_lower_court_id,cl_lower_court_name,cl_lower_court_type
0,United States Court of Appeals for the Ninth C...,ca9,Court of Appeals for the Ninth Circuit,Federal Appellate
1,United States Court of Appeals for the Fifth C...,ca5,Court of Appeals for the Fifth Circuit,Federal Appellate
2,United States Court of Appeals for the Elevent...,ca11,United States Court of Appeals for the Elevent...,Federal Appellate
3,United States Court of Appeals for the Fourth ...,ca4,Court of Appeals for the Fourth Circuit,Federal Appellate
4,United States Court of Appeals for the Sixth C...,ca6,Court of Appeals for the Sixth Circuit,Federal Appellate


# Filter for only Federal Appellate courts

In [8]:
courts_summary = courts_summary[courts_summary["cl_lower_court_type"] == "Federal Appellate"]
len(courts_summary)

13

In [9]:
courts_list = sorted(courts_summary["cl_lower_court_id"].drop_duplicates().astype(str).to_list())
courts_list

['ca1',
 'ca10',
 'ca11',
 'ca2',
 'ca3',
 'ca4',
 'ca5',
 'ca6',
 'ca7',
 'ca8',
 'ca9',
 'cadc',
 'cafc']

In [10]:
sorted(courts_summary["cl_lower_court_name"].drop_duplicates().astype(str).to_list())

['Court of Appeals for the D.C. Circuit',
 'Court of Appeals for the Eighth Circuit',
 'Court of Appeals for the Federal Circuit',
 'Court of Appeals for the Fifth Circuit',
 'Court of Appeals for the First Circuit',
 'Court of Appeals for the Fourth Circuit',
 'Court of Appeals for the Ninth Circuit',
 'Court of Appeals for the Second Circuit',
 'Court of Appeals for the Seventh Circuit',
 'Court of Appeals for the Sixth Circuit',
 'Court of Appeals for the Tenth Circuit',
 'Court of Appeals for the Third Circuit',
 'United States Court of Appeals for the Eleventh Circuit']

# Load CL lower courts data for the Federal Appellate courts

The CL data is extracted from CLReplica for all the lower courts listed

In [11]:
folder = "data/cl_lower_courts"
all_data = []

for court in courts_list:
    filename = f"cl_{court}_dockets.json"
    with open(os.path.join(folder, filename), "r") as f:
        data = json.load(f)
        for record in data:
            record["id"] = court
        all_data.extend(data)  # append list of dicts

# convert to DataFrame
cl_courts = pd.DataFrame(all_data)
cl_courts = cl_courts.add_prefix("cl_lower_court_")
cl_courts.head()

,cl_lower_court_docket_id,cl_lower_court_docket_number,cl_lower_court_cluster_id,cl_lower_court_case_name,cl_lower_court_date_filed,cl_lower_court_precedential_status,cl_lower_court_id
0,2335,11-1921,796936,Mann v. Boatright,2007-02-15,Published,ca1
1,45049,85-1496,463884,Colman v. College Retirement Equities Fund,1985-12-12,Published,ca1
2,54690,96-1186,196969,"Kelleher v. Imaging Systems, Inc",1996-09-06,Unpublished,ca1
3,18837,19-1496,200480,Mason v. Official Committee of Unsecured Credi...,2003-05-27,Published,ca1
4,66092,87-1982_1,513503,"In Re D.C. Sullivan and Company, Inc",1988-08-03,Published,ca1


In [12]:
len(cl_courts)

1359173

## Remove all records that do not have a docket number as they do not contain the info needed to link

In [13]:
cl_courts = cl_courts[~cl_courts["cl_lower_court_docket_number"].isnull()]
cl_courts = cl_courts[cl_courts["cl_lower_court_docket_number"] != ""]
len(cl_courts)

1357376

## Clean up the cl_lower_court_docket_number

In [14]:
cl_courts["cl_lower_court_docket_number_clean"] = cl_courts["cl_lower_court_docket_number"].apply(clean_docket_numbers_fed_appellate)

In [15]:
cl_courts_exploded = cl_courts.explode("cl_lower_court_docket_number_clean", ignore_index=True)
cl_courts_exploded.head()

,cl_lower_court_docket_id,cl_lower_court_docket_number,cl_lower_court_cluster_id,cl_lower_court_case_name,cl_lower_court_date_filed,cl_lower_court_precedential_status,cl_lower_court_id,cl_lower_court_docket_number_clean
0,2335,11-1921,796936,Mann v. Boatright,2007-02-15,Published,ca1,11-1921
1,45049,85-1496,463884,Colman v. College Retirement Equities Fund,1985-12-12,Published,ca1,85-1496
2,54690,96-1186,196969,"Kelleher v. Imaging Systems, Inc",1996-09-06,Unpublished,ca1,96-1186
3,18837,19-1496,200480,Mason v. Official Committee of Unsecured Credi...,2003-05-27,Published,ca1,19-1496
4,66092,87-1982_1,513503,"In Re D.C. Sullivan and Company, Inc",1988-08-03,Published,ca1,87-1982_1


In [16]:
len(cl_courts_exploded[["cl_lower_court_id", "cl_lower_court_docket_number_clean"]].drop_duplicates())

1226493

In [17]:
cl_courts_exploded["cl_lower_court_cluster_id"].nunique()

1357376

In [18]:
cl_courts_exploded = tag_low_ngram_overlap(cl_courts_exploded)

In [19]:
cl_courts_exploded.head()

,cl_lower_court_docket_id,cl_lower_court_docket_number,cl_lower_court_cluster_id,cl_lower_court_case_name,cl_lower_court_date_filed,cl_lower_court_precedential_status,cl_lower_court_id,cl_lower_court_docket_number_clean,low_ngram_overlap
0,2335,11-1921,796936,Mann v. Boatright,2007-02-15,Published,ca1,11-1921,True
1,45049,85-1496,463884,Colman v. College Retirement Equities Fund,1985-12-12,Published,ca1,85-1496,True
2,54690,96-1186,196969,"Kelleher v. Imaging Systems, Inc",1996-09-06,Unpublished,ca1,96-1186,False
3,18837,19-1496,200480,Mason v. Official Committee of Unsecured Credi...,2003-05-27,Published,ca1,19-1496,False
4,66092,87-1982_1,513503,"In Re D.C. Sullivan and Company, Inc",1988-08-03,Published,ca1,87-1982_1,False


In [20]:
cl_courts_exploded = cl_courts_exploded[cl_courts_exploded["low_ngram_overlap"] == False]
len(cl_courts_exploded)

1302280

In [21]:
len(cl_courts_exploded[["cl_lower_court_id", "cl_lower_court_docket_number_clean"]].drop_duplicates())

1161918

In [22]:
cl_courts_exploded["cl_lower_court_cluster_id"].nunique()

1202164

## Inspect duplicates from each circuit courts

In [23]:
for court in courts_list:
    print("----------")
    dups = cl_courts_exploded[cl_courts_exploded["cl_lower_court_id"] == court][["cl_lower_court_id", "cl_lower_court_docket_number_clean"]].value_counts()
    display(dups[dups > 1])

----------


cl_lower_court_id  cl_lower_court_docket_number_clean
ca1                02-1852                               8
                   05-2522                               7
                   07-2159                               7
                   05-2877                               7
                   01-1543                               6
                                                        ..
                   90-1359                               2
                   00-2400                               2
                   01-1056                               2
                   7065                                  2
                   96-2361                               2
Name: count, Length: 5294, dtype: int64

----------


cl_lower_court_id  cl_lower_court_docket_number_clean
ca10               82-2110                               6
                   06-4057                               6
                   86-1400                               6
                   95-3373                               5
                   02-2323                               5
                                                        ..
                   97-5069                               2
                   97-3352                               2
                   97-5056                               2
                   98-3146                               2
                   01-2378                               2
Name: count, Length: 4829, dtype: int64

----------


...

cl_lower_court_id  cl_lower_court_docket_number_clean
ca9                07-15763                              17
                   02-50355                              13
                   00-99005                              12
                   02-56256                              12
                   05-76201                              11
                                                         ..
                   02-30081                               2
                   02-10204                               2
                   03-70185                               2
                   19-16953                               2
                   18024                                  2
Name: count, Length: 19737, dtype: int64

----------


cl_lower_court_id  cl_lower_court_docket_number_clean
cadc               Part I                                8
                   09-3071                               7
                   08-3012                               7
                   95-1075                               7
                   85-1515                               7
                                                        ..
                   06-3156                               2
                   06-5124                               2
                   24871                                 2
                   25-5028                               2
                   06-3150                               2
Name: count, Length: 3319, dtype: int64

----------


cl_lower_court_id  cl_lower_court_docket_number_clean
cafc               2009-3263                             15
                   2010-1548                             12
                   2008-5094                             12
                   2008-3260                             12
                   2009-3127                             12
                                                         ..
                   23-1638                                2
                   94-3415                                2
                   23-1197                                2
                   Misc. 982                              2
                   23-131                                 2
Name: count, Length: 6455, dtype: int64

# Load SCOTUS Scraped Data

In [24]:
scotus = pd.read_json("data/scotus_scraped_data.json")
scotus.head()

,docket_number,filename,docket_date,case_title,lower_court,lower_court_case_numbers_raw,lower_court_case_numbers,lower_court_decision_date,lower_court_rehearing_denied_date,year,case_num
0,03-6084,03-6084.htm,2003-08-27,"Ronald Lee Smith, Petitioner v. Larry Reid, Wa...",United States Court of Appeals for the Tenth C...,(03-1016),03-1016,2003-05-16,2003-07-09,3,6084
1,03-10616,03-10616.htm,2004-05-28,"Jimmy Walker, Petitioner v. Florida","District Court of Appeal of Florida, Fourth Di...",(4D02-4272),4D02-4272,2004-04-21,None,3,10616
2,03-777,03-777.htm,2003-11-28,"Willie R. Flint, Petitioner v. ABB Inc., fka A...",United States Court of Appeals for the Elevent...,(02-15029),02-15029,2003-07-21,2003-08-27,3,777
3,03-10170,03-10170.htm,2004-05-05,"Loyda Lugones, Petitioner v. United States",United States Court of Appeals for the Elevent...,(02-12984),02-12984,2004-01-26,None,3,10170
4,03-8917,03-8917.htm,2004-02-19,"Ray Charles Smith, Petitioner v. United States",United States Court of Appeals for the Ninth C...,(03-10003),03-10003,2003-11-13,None,3,8917


## Remove the scotus records that do not have valid lower court info for appellate chain linking

In [25]:
scotus = scotus[(~scotus["lower_court"].isnull()) & (~scotus["lower_court_case_numbers"].isnull())]
len(scotus)

185843

# Filter for those from the Federal Appellate courts

In [26]:
scotus.columns

Index(['docket_number', 'filename', 'docket_date', 'case_title', 'lower_court',
       'lower_court_case_numbers_raw', 'lower_court_case_numbers',
       'lower_court_decision_date', 'lower_court_rehearing_denied_date',
       'year', 'case_num'],
      dtype='object')

In [27]:
scotus = scotus[scotus["lower_court"].isin(list(courts_summary["scotus_lower_court"].unique()))]
len(scotus)

142944

# Clean-up the SCOTUS scotus_lower_court_case_numbers and explode it

In [28]:
scotus = scotus.add_prefix('scotus_')

In [29]:
scotus['scotus_lower_court_case_number'] = scotus['scotus_lower_court_case_numbers'].str.split(',')
scotus_exploded = scotus.explode('scotus_lower_court_case_number', ignore_index=True)
len(scotus_exploded)

156941

In [30]:
scotus_exploded['scotus_lower_court_case_number'] = scotus_exploded["scotus_lower_court_case_number"].apply(clean_scotus_lower_case_nums)
scotus_exploded = scotus_exploded[(~scotus_exploded["scotus_lower_court_case_number"].isnull()) & (scotus_exploded["scotus_lower_court_case_number"]!= "")]

scotus_exploded[["scotus_lower_court", "scotus_lower_court_case_number"]].value_counts()

scotus_lower_court                                                   scotus_lower_court_case_number
United States Court of Appeals for the Sixth Circuit                 21-7000                           15
United States Court of Appeals for the Eighth Circuit                14-2220                           15
United States Court of Appeals for the District of Columbia Circuit  09-1322                           14
                                                                     06-5268                           11
                                                                     06-5267                           11
                                                                                                       ..
United States Court of Appeals for the Fifth Circuit                 23-10459                           1
                                                                     23-10458                           1
                                                    

In [31]:
len(scotus_exploded[["scotus_lower_court", "scotus_lower_court_case_number"]].drop_duplicates())

136865

# Join cl_courts_exploded to courts_summary to map each cl_lower_court_id to scotus_lower_court

In [32]:
len(cl_courts_exploded)

1302280

In [33]:
courts_bridge = pd.merge(cl_courts_exploded, courts_summary, on="cl_lower_court_id", how="left")
len(courts_bridge)

1302280

In [34]:
## Since there is a one-to-one match between cl_lower_court_id and scotus_lower_court for circuit courts, 
## the number of records before and after the join should remain the same
assert len(cl_courts_exploded) == len(courts_bridge)

# Join scotus_exploded to courts_bridge by court and docket_number to establish an appellate chain

In [35]:
len(scotus_exploded)

156897

In [36]:
app_chain = pd.merge(scotus_exploded, courts_bridge, left_on=["scotus_lower_court", "scotus_lower_court_case_number"], right_on=["scotus_lower_court", "cl_lower_court_docket_number_clean"], how="left")
len(app_chain)

180019

In [37]:
missing_chain = app_chain[app_chain["cl_lower_court_docket_number_clean"].isnull()]
len(missing_chain)

65369

In [38]:
missing_chain[["scotus_docket_number", "scotus_lower_court", "scotus_lower_court_case_number"]]

,scotus_docket_number,scotus_lower_court,scotus_lower_court_case_number
0,03-6084,United States Court of Appeals for the Tenth C...,03-1016
2,03-10170,United States Court of Appeals for the Elevent...,02-12984
4,03-10164,United States Court of Appeals for the Elevent...,03-11101-AA
18,03-8095,United States Court of Appeals for the Ninth C...,01-56837
28,03-6721,United States Court of Appeals for the Third C...,02-1352
...,...,...,...
180008,22A202,United States Court of Appeals for the Federal...,2021-1378
180013,22A594,United States Court of Appeals for the Second ...,22-107
180014,22A255,United States Court of Appeals for the Sixth C...,20-4303
180016,22A469,United States Court of Appeals for the Ninth C...,22-15480


In [39]:
courts_bridge[courts_bridge["cl_lower_court_docket_number_clean"] == "22-107-cv"]

,cl_lower_court_docket_id,cl_lower_court_docket_number,cl_lower_court_cluster_id,cl_lower_court_case_name,cl_lower_court_date_filed,cl_lower_court_precedential_status,cl_lower_court_id,cl_lower_court_docket_number_clean,low_ngram_overlap,scotus_lower_court,cl_lower_court_name,cl_lower_court_type
259708,65423959,22-107-cv,8248873,"United States Ex Rel. Yu v. Grifols USA, LLC",2022-10-14,Unpublished,ca2,22-107-cv,False,United States Court of Appeals for the Second ...,Court of Appeals for the Second Circuit,Federal Appellate


In [40]:
app_chain = app_chain[~app_chain["cl_lower_court_docket_number_clean"].isnull()]
len(app_chain)

114650

In [41]:
app_chain["scotus_docket_number"].nunique()

84741

In [42]:
len(app_chain[["cl_lower_court_id", "cl_lower_court_docket_number_clean"]].drop_duplicates())

79196

In [43]:
app_chain["scotus_lower_court_decision_date"].isnull().sum()

np.int64(26233)

In [44]:
app_chain["cl_lower_court_date_filed"].isnull().sum()

np.int64(0)

In [45]:
len(app_chain[app_chain["scotus_lower_court_decision_date"] == app_chain["cl_lower_court_date_filed"]][["cl_lower_court_id", "cl_lower_court_docket_number_clean"]].drop_duplicates())

65954

In [46]:
app_chain.columns

Index(['scotus_docket_number', 'scotus_filename', 'scotus_docket_date',
       'scotus_case_title', 'scotus_lower_court',
       'scotus_lower_court_case_numbers_raw',
       'scotus_lower_court_case_numbers', 'scotus_lower_court_decision_date',
       'scotus_lower_court_rehearing_denied_date', 'scotus_year',
       'scotus_case_num', 'scotus_lower_court_case_number',
       'cl_lower_court_docket_id', 'cl_lower_court_docket_number',
       'cl_lower_court_cluster_id', 'cl_lower_court_case_name',
       'cl_lower_court_date_filed', 'cl_lower_court_precedential_status',
       'cl_lower_court_id', 'cl_lower_court_docket_number_clean',
       'low_ngram_overlap', 'cl_lower_court_name', 'cl_lower_court_type'],
      dtype='object')

# Save the data

In [47]:
app_chain.to_csv("data/joined_circuit.csv", index=False)